In [1]:
import pandas as pd
import numpy as np

from google.colab import files
uploaded = files.upload()

CSV_PATH = "/content/neo_asteroid_dataset.csv"
df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nPHA counts:")
print(df["pha"].value_counts(dropna=False))

print("\nMissing values:")
print(df.isna().sum().sort_values(ascending=False))

print("\nFirst five rows:")
display(df.head())

Saving neo_asteroid_dataset.csv to neo_asteroid_dataset.csv
Dataset shape: (42815, 21)

Columns:
['spkid', 'pdes', 'full_name', 'kind', 'neo', 'pha', 'e', 'a', 'q', 'i', 'om', 'w', 'moid', 'H', 'diameter', 'albedo', 'first_obs', 'last_obs', 'data_arc', 'n_obs_used', 'condition_code']

PHA counts:
pha
N      40130
Y       2550
NaN      135
Name: count, dtype: int64

Missing values:
albedo            41611
diameter          41569
data_arc            437
pha                 135
moid                135
H                     9
condition_code        2
neo                   0
pdes                  0
spkid                 0
kind                  0
full_name             0
a                     0
w                     0
om                    0
i                     0
q                     0
e                     0
first_obs             0
last_obs              0
n_obs_used            0
dtype: int64

First five rows:


,spkid,pdes,full_name,kind,neo,pha,e,a,q,i,...,w,moid,H,diameter,albedo,first_obs,last_obs,data_arc,n_obs_used,condition_code
0,20000433,433,433 Eros (A898 PA),an,Y,N,0.2229,1.458,1.133,10.83,...,178.92,0.1490,10.40,16.840,0.250,1893-10-29,2021-05-13,46582.0,9130,0.0
1,20000719,719,719 Albert (A911 TB),an,Y,N,0.5466,2.637,1.196,11.57,...,156.18,0.2020,15.59,NaN,NaN,1911-10-04,2026-05-22,41869.0,2147,0.0
2,20000887,887,887 Alinda (A918 AA),an,Y,N,0.5711,2.474,1.061,9.40,...,350.55,0.0801,13.82,4.200,0.310,1918-02-09,2025-08-27,39281.0,5843,0.0
3,20001036,1036,1036 Ganymed (A924 UB),an,Y,N,0.5335,2.664,1.243,26.69,...,132.51,0.3420,9.17,37.675,0.238,1924-10-23,2026-06-30,37140.0,13637,0.0
4,20001221,1221,1221 Amor (1932 EA1),an,Y,N,0.4347,1.920,1.085,11.87,...,26.75,0.1090,17.37,1.000,NaN,1932-03-12,2026-10-20,34555.0,737,0.0


In [2]:
# Select the features used for the experiment
FEATURES = ["a", "e", "i", "om", "w", "diameter"]
TARGET = "pha"

# Check required columns
missing_columns = [
    col for col in FEATURES + [TARGET]
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing columns in dataset: {missing_columns}")

# Keep rows with known target labels
data = df.dropna(subset=[TARGET]).copy()

# Convert target labels to 0 and 1
data[TARGET] = (
    data[TARGET]
    .astype(str)
    .str.strip()
    .str.upper()
    .map({"Y": 1, "N": 0})
)

# Remove rows with unknown/unrecognized target labels
data = data.dropna(subset=[TARGET]).copy()
data[TARGET] = data[TARGET].astype(int)

# Convert features to numeric
for col in FEATURES:
    data[col] = pd.to_numeric(data[col], errors="coerce")

X = data[FEATURES]
y = data[TARGET]

print("Prepared dataset shape:", data.shape)
print("\nTarget distribution:")
print(y.value_counts())

print("\nMissing values in selected features:")
print(X.isna().sum())

Prepared dataset shape: (42680, 21)

Target distribution:
pha
0    40130
1     2550
Name: count, dtype: int64

Missing values in selected features:
a               0
e               0
i               0
om              0
w               0
diameter    41434
dtype: int64


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nMissing diameter values in training set:",
      X_train["diameter"].isna().sum())

print("Missing diameter values in test set:",
      X_test["diameter"].isna().sum())

Training rows: 34144
Testing rows: 8536

Missing diameter values in training set: 33145
Missing diameter values in test set: 8289


In [4]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

print("Method 2 model trained successfully!")
print("Imputation: Median")
print("Missingness indicators: Not added")
print("Classifier: HistGradientBoosting")

Method 2 model trained successfully!
Imputation: Median
Missingness indicators: Not added
Classifier: HistGradientBoosting


In [5]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

results = {
    "Method": "Median Imputation",
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1-score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob),
    "Average Precision": average_precision_score(y_test, y_prob)
}

print("METHOD 2: MEDIAN IMPUTATION")
print("-" * 40)

for metric, value in results.items():
    if isinstance(value, (int, float, np.integer, np.floating)):
        print(f"{metric}: {value:.4f}")
    else:
        print(f"{metric}: {value}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Non-PHA", "PHA"],
    zero_division=0
))

METHOD 2: MEDIAN IMPUTATION
----------------------------------------
Method: Median Imputation
Accuracy: 0.7433
Precision: 0.1511
Recall: 0.7137
F1-score: 0.2494
ROC-AUC: 0.8106
Average Precision: 0.2474

Confusion Matrix:
[[5981 2045]
 [ 146  364]]

Classification Report:
              precision    recall  f1-score   support

     Non-PHA       0.98      0.75      0.85      8026
         PHA       0.15      0.71      0.25       510

    accuracy                           0.74      8536
   macro avg       0.56      0.73      0.55      8536
weighted avg       0.93      0.74      0.81      8536



In [6]:
results_df = pd.DataFrame([results])

results_df.to_csv(
    "method2_median_imputation_results.csv",
    index=False
)

results_df

,Method,Accuracy,Precision,Recall,F1-score,ROC-AUC,Average Precision
0,Median Imputation,0.743322,0.1511,0.713725,0.2494,0.810588,0.247431
